# gold_dec_fec

Camada **Gold** de continuidade do fornecimento: DEC e FEC por alimentador e mês, para a tela de Operação.

**O que são DEC e FEC.** São os dois indicadores que a ANEEL usa para medir a qualidade do fornecimento de energia:
- **DEC** é quantas horas, em média, cada consumidor ficou sem energia no período. DEC 2,5 no mês significa que cada UC do alimentador ficou, em média, 2h30 sem energia.
- **FEC** é quantas vezes, em média, cada consumidor ficou sem energia. FEC 0,5 no mês significa, em média, uma interrupção a cada duas UCs por mês.

O DEC mede quanto tempo a energia faltou, e o FEC mede quantas vezes. Para um alimentador num mês, DEC = soma das horas de interrupção ÷ UCs do alimentador, e FEC = número de interrupções ÷ UCs do alimentador. Como cada linha da Silver já é uma UC interrompida, basta somar as durações e contar as linhas.

**O que este notebook faz.** Resume as 180 mil interrupções da Silver numa tabela com uma linha por alimentador e mês. A tela de Operação e a IA leem essa tabela pronta para mostrar os piores alimentadores, a evolução no tempo e a comparação entre municípios.

- **Entrada:** `gold.dim_mes`, `gold.dim_alimentador` e `gold.dim_uc` (`gold_base`) e `silver.tb_ocorrencias_rede` (`silver_rede_atendimento`).
- **Saída:** `gold.agg_dec_fec_mensal`.
- **Alimenta:** `GET /operacao` (série histórica, ranking de alimentadores e ranking regional) e o agente de IA.
- Segue o documento *Decisões da Engenharia de Dados: camada Gold* (v2, seções 4 e 5.1). Idempotente: grava em overwrite e pode ser reexecutado.

| Tabela | Grão (1 linha por) | Conteúdo |
|---|---|---|
| `agg_dec_fec_mensal` | alimentador × mês (122 × 48) | município, período (P1/P2), UCs atendidas, interrupções, horas de interrupção, DEC, FEC e contagens das flags da Silver |

**Regras**
- **Fórmula (PRODIST Módulo 8):** DEC = Σ t(i) ÷ Cc e FEC = Σ Ca(i) ÷ Cc. Cada linha da Silver é uma UC interrompida, então Ca(i) = 1 e t(i) = duração em horas. Cc = `ucs_atendidas` do alimentador na `dim_alimentador`.
- **Mesma definição do `gold_kpis`:** mês pela data de abertura e só interrupções acima de 3 minutos. A soma dos alimentadores reproduz o DEC/FEC global.
- **Flags da Silver:** interrupções sobrepostas e no teto de 48h entram no cálculo padrão. As contagens ficam expostas por alimentador e mês, junto com as acima de 24h.
- **Mês sem interrupção** tem 0 interrupções e DEC/FEC 0. Não é dado faltando.
- Antes de gravar: unicidade do grão, chaves sem nulos, grade completa e reconciliação com a Silver. Se algo falhar, nada é gravado.

## Funções comuns
**O que fazemos:** carregamos as funções compartilhadas pelo time.

In [0]:
%run ../bronze-to-silver/funcoes_comuns

## Configuração
**O que fazemos:** lemos o catálogo do widget.

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

## Parâmetros
**O que fazemos:** reunimos os parâmetros do cálculo, cada um com o que representa e o seu valor no contexto real. Mudar uma regra é mudar só esta célula.

In [0]:
# Duração mínima para a interrupção contar no DEC/FEC.
# Real: o PRODIST Módulo 8 só considera interrupções longas, acima de 3 minutos.
# Nos dados, a menor interrupção tem 5 minutos, então nenhuma sai, mas a regra fica no código.
DURACAO_MINIMA_DEC_MIN = 3

# Minutos por hora: o DEC é medido em horas por UC, e a Silver guarda a duração em minutos.
MINUTOS_POR_HORA = 60

# Grão da tabela: 1 linha por alimentador e mês.
CHAVE = ["alimentador", "mes_referencia"]

# Contagens das flags da Silver expostas por alimentador e mês (entram no cálculo, ficam visíveis à parte).
COLUNAS_FLAGS = ["ocorrencias_sobrepostas", "ocorrencias_teto_48h", "ocorrencias_acima_24h"]

# Ordem final das colunas (nomes iguais aos do contrato da API).
COLUNAS = [*CHAVE, "municipio", "periodo", "ucs_atendidas", "interrupcoes", "horas_interrupcao",
           "dec", "fec", *COLUNAS_FLAGS]

## Leitura
**O que fazemos:** lemos as dimensões da `gold_base` e as ocorrências da Silver.

In [0]:
dim_mes = spark.table(f"{CATALOGO}.gold.dim_mes").select("mes_referencia", "periodo")
dim_alim = spark.table(f"{CATALOGO}.gold.dim_alimentador").select("alimentador", "municipio", "ucs_atendidas")
dim_uc = spark.table(f"{CATALOGO}.gold.dim_uc")
oc = spark.table(f"{CATALOGO}.silver.tb_ocorrencias_rede")

N_MESES, N_ALIM, N_UCS, N_OC = dim_mes.count(), dim_alim.count(), dim_uc.count(), oc.count()
print(f"meses: {N_MESES} | alimentadores: {N_ALIM} | UCs: {N_UCS:,} | ocorrências na Silver: {N_OC:,}")

**O que descobrimos:**
- As entradas têm o tamanho esperado: 48 meses, 122 alimentadores, 45.000 UCs e 180.000 ocorrências na Silver.

## Pré-condições das entradas
**O que fazemos:** conferimos que as dimensões estão completas antes de calcular. O Cc de cada alimentador não pode ser zero ou nulo, e a soma dos Cc tem que ser o total de UCs do cadastro. Se algo falhar, o problema está na `gold_base`, e não aqui.

In [0]:
checar_unicidade(dim_alim, ["alimentador"], "dim_alimentador")
checar_unicidade(dim_mes, ["mes_referencia"], "dim_mes")
checar_sem_nulos(dim_alim, ["alimentador", "municipio", "ucs_atendidas"], "dim_alimentador")

sem_ucs = dim_alim.filter(F.col("ucs_atendidas") <= 0).count()
assert sem_ucs == 0, f"{sem_ucs} alimentadores com Cc zero ou negativo: o DEC deles seria uma divisão por zero"

soma_cc = dim_alim.agg(F.sum("ucs_atendidas")).first()[0]
assert soma_cc == N_UCS, f"Σ ucs_atendidas = {soma_cc:,}, mas a dim_uc tem {N_UCS:,}"

print(f"ok: {N_ALIM} alimentadores, Cc de {dim_alim.agg(F.min('ucs_atendidas')).first()[0]} "
      f"a {dim_alim.agg(F.max('ucs_atendidas')).first()[0]} UCs, soma {soma_cc:,} = dim_uc")

**O que descobrimos:**
- As dimensões estão completas: alimentadores e meses únicos, sem nulos.
- O Cc varia muito entre alimentadores, de 20 a 2.285 UCs. A soma dá exatamente as 45.000 UCs da `dim_uc`.

## Ocorrências válidas
**O que fazemos:** ficamos só com as interrupções acima de 3 minutos e definimos o mês pela data de abertura, como no `gold_kpis`.

In [0]:
oc_validas = (oc.filter(F.col("duracao_min") > DURACAO_MINIMA_DEC_MIN)
                .withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM")))
N_VALIDAS = oc_validas.count()
print(f"ocorrências válidas: {N_VALIDAS:,} | fora do critério (≤ {DURACAO_MINIMA_DEC_MIN} min): {N_OC - N_VALIDAS:,}")

**O que descobrimos:**
- As 180.000 ocorrências passam no critério do PRODIST (acima de 3 minutos). Nenhuma sai, porque a menor dura 5 minutos.

## Integridade com as dimensões
**O que fazemos:** conferimos que todo alimentador e todo mês das ocorrências existem nas dimensões. Se algum faltar, aquelas interrupções sumiriam do DEC sem aviso.

In [0]:
alim_fora = oc_validas.join(dim_alim, "alimentador", "left_anti")
mes_fora = oc_validas.join(dim_mes, "mes_referencia", "left_anti")
n_alim_fora, n_mes_fora = alim_fora.count(), mes_fora.count()

if n_alim_fora:
    alim_fora.groupBy("alimentador").count().show(truncate=False)
if n_mes_fora:
    mes_fora.groupBy("mes_referencia").count().show(truncate=False)
assert n_alim_fora == 0, f"{n_alim_fora:,} ocorrências com alimentador fora da dim_alimentador"
assert n_mes_fora == 0, f"{n_mes_fora:,} ocorrências com mês fora da dim_mes"
print("ok: todo alimentador e todo mês das ocorrências existem nas dimensões")

**O que descobrimos:**
- Todo alimentador e todo mês das ocorrências existem nas dimensões. Nenhuma interrupção fica de fora do cálculo.

## Agregação por alimentador e mês
**O que fazemos:** somamos, para cada alimentador e mês, o número de interrupções e os minutos interrompidos. Também contamos as interrupções marcadas pelas flags da Silver.

In [0]:
conta = lambda flag: F.sum(F.col(flag).cast("long"))

agregado = (oc_validas.groupBy(*CHAVE)
    .agg(F.count("*").alias("interrupcoes"),
         F.sum("duracao_min").alias("__minutos"),
         conta("flag_sobreposicao").alias("ocorrencias_sobrepostas"),
         conta("flag_duracao_teto").alias("ocorrencias_teto_48h"),
         conta("flag_longa_duracao").alias("ocorrencias_acima_24h")))
print(f"pares alimentador × mês com interrupção: {agregado.count():,} de {N_ALIM * N_MESES:,} possíveis")

**O que descobrimos:**
- Todos os 5.856 pares de alimentador e mês têm pelo menos uma interrupção. A grade completa não precisa preencher nenhum mês com zero, mas a regra fica no código.

## Grade completa e cálculo do DEC e FEC
**O que fazemos:** cruzamos todos os alimentadores com todos os meses, para que mês sem interrupção apareça com 0. Depois calculamos as horas, o DEC e o FEC, sempre a partir dos minutos (sem arredondar antes) e com `try_divide`.

In [0]:
dec_fec = (dim_alim.crossJoin(dim_mes)
    .join(agregado, CHAVE, "left")
    .fillna(0, subset=["interrupcoes", "__minutos", *COLUNAS_FLAGS])
    .withColumns({
        "horas_interrupcao": F.try_divide(F.col("__minutos"), F.lit(MINUTOS_POR_HORA)).cast("decimal(14,2)"),
        "dec": F.try_divide(F.col("__minutos"), F.col("ucs_atendidas") * MINUTOS_POR_HORA).cast("decimal(14,4)"),
        "fec": F.try_divide(F.col("interrupcoes"), F.col("ucs_atendidas")).cast("decimal(14,4)"),
    })
    .select(*COLUNAS, "__minutos"))

## Checagens e reconciliação
**O que fazemos:** conferimos o grão, os nulos e a grade completa. A reconciliação garante que as interrupções, os minutos e as flags da Gold são iguais aos da Silver. Se algo falhar, nada é gravado.

In [0]:
checar_unicidade(dec_fec, CHAVE, "agg_dec_fec_mensal")
checar_sem_nulos(dec_fec, [*CHAVE, "municipio", "ucs_atendidas", "interrupcoes",
                           "horas_interrupcao", "dec", "fec"], "agg_dec_fec_mensal")

n_linhas = dec_fec.count()
assert n_linhas == N_ALIM * N_MESES, f"esperado {N_ALIM * N_MESES:,} linhas, veio {n_linhas:,}"

g = dec_fec.agg(F.sum("interrupcoes").alias("n"), F.sum("__minutos").alias("min"),
                *[F.sum(c).alias(c) for c in COLUNAS_FLAGS]).first().asDict()
s = oc_validas.agg(F.count("*").alias("n"), F.sum("duracao_min").alias("min"),
                   conta("flag_sobreposicao").alias("ocorrencias_sobrepostas"),
                   conta("flag_duracao_teto").alias("ocorrencias_teto_48h"),
                   conta("flag_longa_duracao").alias("ocorrencias_acima_24h")).first().asDict()
assert g == s, f"reconciliação não fecha: gold={g} silver={s}"

print(f"ok: {n_linhas:,} linhas | interrupções {g['n']:,} | horas {g['min'] / MINUTOS_POR_HORA:,.2f} | "
      f"sobrepostas {g['ocorrencias_sobrepostas']} | teto 48h {g['ocorrencias_teto_48h']} | "
      f"acima 24h {g['ocorrencias_acima_24h']}: igual à Silver")

**O que descobrimos:**
- O grão é único, as chaves não têm nulos e a tabela tem as 5.856 linhas esperadas.
- A reconciliação fecha com a Silver: 180.000 interrupções, 356.534,78 horas, 331 sobrepostas, 30 no teto de 48h e 274 acima de 24h.

## Gravação
**O que fazemos:** gravamos a tabela em overwrite, com a coluna de rastreio `_atualizado_em`.

In [0]:
(dec_fec.drop("__minutos")
        .withColumn("_atualizado_em", F.current_timestamp())
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOGO}.gold.agg_dec_fec_mensal"))

## Conferência da tabela gravada
**O que fazemos:** lemos a tabela de volta e conferimos o número de linhas, os tipos e uma amostra.

In [0]:
gold = spark.table(f"{CATALOGO}.gold.agg_dec_fec_mensal")
print(f"gold.agg_dec_fec_mensal: {gold.count():,} linhas")
gold.printSchema()
gold.orderBy("alimentador", "mes_referencia").show(5, truncate=False)

**O que descobrimos:**
- A tabela foi gravada com 5.856 linhas e os tipos do padrão da Gold: DEC e FEC em decimal(14,4), horas em decimal(14,2) e contagens inteiras.

## DEC e FEC global por período
**O que fazemos:** somamos os alimentadores para obter o DEC e o FEC da distribuidora em cada período. No período, DEC e FEC são a soma dos meses, como no contrato da API. O resultado deve bater com o `gold_kpis`.

In [0]:
(gold.filter(F.col("periodo").isNotNull()).groupBy("periodo")
     .agg(F.sum("interrupcoes").alias("interrupcoes"),
          F.sum("horas_interrupcao").alias("horas_interrupcao"),
          F.round(F.sum("horas_interrupcao") / N_UCS, 4).alias("dec_global"),
          F.round(F.sum("interrupcoes") / N_UCS, 4).alias("fec_global"))
     .orderBy("periodo").show(truncate=False))

print("== DEC global por mês (todos os meses)")
(gold.groupBy("mes_referencia")
     .agg(F.round(F.sum("horas_interrupcao") / N_UCS, 4).alias("dec_global"),
          F.round(F.sum("interrupcoes") / N_UCS, 4).alias("fec_global"))
     .orderBy("mes_referencia").show(48, truncate=False))

**O que descobrimos:**
- Um mês normal tem DEC de cerca de 0,12 h/UC e FEC de cerca de 0,075. Cada UC fica, em média, uns 7 minutos sem energia por mês.
- Do P1 para o P2, o DEC global sobe de 1,47 para 3,45 h/UC (+134%) e o FEC de 0,89 para 1,31 (+46%).
- Essa piora vem quase toda de julho de 2025, com DEC de 1,94 h/UC, 16 vezes um mês normal. Sem julho, o P2 teria DEC de 1,51 (+2% sobre o P1) e FEC de 0,91 (+2%). Ou seja, fora o evento extremo, a qualidade do fornecimento ficou estável.
- Novembro de 2025 e fevereiro de 2026 também aparecem acima do normal (DEC de cerca de 0,19), pelo excesso de ocorrências já registrado na Silver.

## Piores e melhores alimentadores
**O que fazemos:** montamos o ranking dos últimos 12 meses (P2), que é o que a tela de Operação mostra por padrão. Listamos os 10 piores e os 5 melhores em DEC e os 10 piores em FEC.

In [0]:
ranking_p2 = (gold.filter(F.col("periodo") == "P2")
    .groupBy("alimentador", "municipio", "ucs_atendidas")
    .agg(F.sum("interrupcoes").alias("interrupcoes"),
         F.sum("dec").alias("dec_p2"), F.sum("fec").alias("fec_p2")))

print("== 10 piores em DEC (P2)")
ranking_p2.orderBy(F.desc("dec_p2")).show(10, truncate=False)
print("== 5 melhores em DEC (P2)")
ranking_p2.orderBy("dec_p2").show(5, truncate=False)
print("== 10 piores em FEC (P2)")
ranking_p2.orderBy(F.desc("fec_p2")).show(10, truncate=False)

**O que descobrimos:**
- Sete dos 10 piores alimentadores em DEC são muito pequenos, com 20 a 47 UCs. Todos têm cerca de 370 a 400 interrupções no ano, como alimentadores grandes, mas divididas por poucas UCs. O pior, ALM-JAB-016, tem só 20 UCs e DEC de 43,9 h/UC.
- Os outros três são os alimentadores do evento de julho de 2025: ALM-SAO-004, ALM-JAB-004 e ALM-REC-008. São grandes (863 a 2.285 UCs) e estão na lista pelo volume de interrupções daquele mês.
- Os melhores são alimentadores grandes, com cerca de 1.000 a 1.300 UCs e o mesmo número de interrupções dos pequenos.
- O ranking de FEC repete quase os mesmos alimentadores pequenos.

## Concentração das interrupções
**O que fazemos:** verificamos se as interrupções acompanham o tamanho do alimentador. Se não acompanharem, alimentadores pequenos ficam com DEC alto só por terem poucas UCs no denominador. Também medimos o peso de julho de 2025 no DEC de cada período.

In [0]:
total_4_anos = (gold.groupBy("alimentador", "ucs_atendidas")
                    .agg(F.sum("interrupcoes").alias("interrupcoes"), F.sum("dec").alias("dec_total")))

print("correlação entre UCs do alimentador e interrupções (4 anos):",
      round(total_4_anos.stat.corr("ucs_atendidas", "interrupcoes"), 3))
print("correlação entre UCs do alimentador e DEC (4 anos):",
      round(total_4_anos.stat.corr("ucs_atendidas", "dec_total"), 3))

print("\n== interrupções por faixa de tamanho do alimentador (4 anos)")
(total_4_anos
    .withColumn("faixa_ucs", F.when(F.col("ucs_atendidas") < 100, "1. menos de 100")
                              .when(F.col("ucs_atendidas") < 300, "2. 100 a 299")
                              .when(F.col("ucs_atendidas") < 600, "3. 300 a 599")
                              .otherwise("4. 600 ou mais"))
    .groupBy("faixa_ucs")
    .agg(F.count("*").alias("alimentadores"),
         F.round(F.avg("ucs_atendidas")).alias("ucs_media"),
         F.round(F.avg("interrupcoes")).alias("interrupcoes_media"),
         F.round(F.avg("dec_total"), 2).alias("dec_medio_4_anos"))
    .orderBy("faixa_ucs").show(truncate=False))

print("== peso de julho de 2025 no DEC global de P2")
p2 = gold.filter(F.col("periodo") == "P2")
dec_p2 = p2.agg(F.sum("horas_interrupcao")).first()[0] / N_UCS
dec_jul = p2.filter(F.col("mes_referencia") == "2025-07").agg(F.sum("horas_interrupcao")).first()[0] / N_UCS
print(f"DEC P2: {dec_p2:.4f} h/UC | só julho de 2025: {dec_jul:.4f} ({dec_jul / dec_p2:.0%} do período)")

**O que descobrimos:**
- Fora os três alimentadores do evento de julho, o número de interrupções quase não depende do tamanho do alimentador: cerca de 1.370 em 4 anos, tanto nos de 60 UCs quanto nos de 440. A correlação positiva (0,62) vem desses três alimentadores grandes.
- Como o DEC divide pelas UCs, os alimentadores pequenos ficam com DEC muito mais alto: 52 h/UC em 4 anos nos menores, contra cerca de 5,7 nos maiores. Por isso o DEC cai quando o alimentador cresce (correlação de −0,41).
- Julho de 2025 sozinho responde por 56% do DEC global do P2.
- Para a tela de Operação, o ranking por DEC destaca os alimentadores pequenos. Para priorizar onde mais consumidores sofrem, vale olhar também as horas de interrupção e o número de UCs.

## Resumo
**O que entregamos:** a tabela `gold.agg_dec_fec_mensal`, com DEC e FEC de cada um dos 122 alimentadores em cada um dos 48 meses (5.856 linhas). Ela alimenta a tela de Operação e o agente de IA.

**Como garantimos que está certa:**
- As entradas estão completas e o Cc de cada alimentador fecha com as 45.000 UCs do cadastro.
- Todas as 180.000 interrupções da Silver entram no cálculo, e a reconciliação fecha com a Silver em interrupções, horas e flags.
- O grão é único, sem nulos, e o cálculo segue o PRODIST e a mesma definição do `gold_kpis`.

**O que os números mostram:**
- Um mês normal tem DEC de cerca de 0,12 h/UC. Do P1 para o P2, o DEC global sobe 134%, mas 56% do DEC do P2 vem só de julho de 2025. Sem esse mês, a qualidade fica estável (+2%).
- As interrupções se distribuem quase igualmente entre os alimentadores, sem acompanhar o tamanho deles. Por isso, os piores em DEC são alimentadores muito pequenos (20 a 47 UCs), além dos três alimentadores do evento de julho.

**Decisões registradas:** interrupções sobrepostas (331) e no teto de 48h (30) entram no cálculo, com as contagens expostas por linha. Julho de 2025, novembro de 2025 e fevereiro de 2026 ficam como estão.